# 1. Обробка файлів в пакеті 

In [9]:
# Parameters
file_cnt = 0


StatementMeta(, 2b4638ce-0775-4bb2-86aa-df0a72e35642, 13, Finished, Available, Finished, False)

## Оброка самих файлів і запис в цю таблицю **dbo.ev2_file_body**


## Сервісні функції для обробки файлів

In [10]:
from pyspark.sql.functions import input_file_name, regexp_extract, col, when, sum, count, substring, lit, coalesce, sequence, expr


def read_files(all_files):
    """
        Прочитати файли по списку і записати в таблицю.
        При запису FILE_ID  співпадає з полем id в журналі файлів
    """
    target_table = "dbo.ev2_file_body"
    print(f"Починаємо обробку {len(all_files)} файлів...")

    # Створюємо порожній DataFrame зі схемою першого файлу (або пустий)
    # Найкращий спосіб — прочитати кожен файл окремо і додати йому ID
    combined_df = None

    for file_info in all_files:
        path = file_info["snapshot_path"]
        context_id = file_info["id"]
        
        # Читаємо конкретний файл
        temp_df = spark.read.json(path)
        
        # Додаємо метадані прямо в DataFrame
        temp_df = temp_df.withColumn("file_name", lit(file_info["file_name"])) \
                         .withColumn("file_id", lit(context_id))
        
        if combined_df is None:
            combined_df = temp_df
        else:
            combined_df = combined_df.unionByName(temp_df, allowMissingColumns=True)

    if combined_df:
        combined_df.createOrReplaceTempView("v_batch_incoming_data")

        # Виконуємо MERGE, тепер додаючи FILE_ID
        spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING v_batch_incoming_data AS source
            ON target.TX_ID = source.TX_ID
            WHEN MATCHED THEN
                UPDATE SET 
                    target.STATUS = 'UPDATED', 
                    target.FILE_NAME = source.file_name,
                    target.FILE_ID = source.file_id
            WHEN NOT MATCHED THEN
                INSERT (TX_ID, TERMINAL_ID, OPDATE, NAME, PASSPORT, AMOUNT, CHARGE, STATUS, FILE_NAME, FILE_ID)
                VALUES (source.TX_ID, source.terminal, source.opdate, source.name, source.passport, 
                        source.amount, source.charge, 'NEW', source.file_name, source.file_id)
        """)

        print("MERGE завершено успішно (з урахуванням FILE_ID).")


def archive_processed_files(all_files):
    """
        Перенесення оброблених файлів в архів
    """
    archive_dir = "Files/archive"
    
    # Створюємо директорію архіву, якщо її немає
    if not mssparkutils.fs.exists(archive_dir):
        mssparkutils.fs.mkdirs(archive_dir)
        
    for file_info in all_files:
        source_path = file_info["source_path"]
        file_name = file_info["file_name"]
        
        # Можна додати дату до шляху для кращої структури: archive/2026-05-04/file.json
        target_path = f"{archive_dir}/{file_name}"
        
        print(f"Переміщую в архів: {file_name}")
        
        # Переміщуємо оригінальний файл (mv замість cp)
        mssparkutils.fs.mv(source_path, target_path)
        
        # Також варто видалити тимчасовий снепшот
        if mssparkutils.fs.exists(file_info["snapshot_path"]):
            mssparkutils.fs.rm(file_info["snapshot_path"], True)


def set_status_processed(all_files):
    """
        ВЗмінити статус файлів на PROCESSED
    """
    for file_info in all_files:
        file_name = file_info["file_name"]
        context_id = file_info["id"]
        # Використовуємо синтаксис :назва_змінної всередині запиту
        spark.sql("""
            UPDATE ev2_file_grn
            SET 
            processing_status = 'PROCESSED',
            processed_at = current_timestamp()
            WHERE id = :context_id AND processing_status = 'NEW'
        """, args={"context_id": context_id})
        print(f"Відмічаю, що файл оброблено: {file_name} - ok")  

StatementMeta(, 2b4638ce-0775-4bb2-86aa-df0a72e35642, 14, Finished, Available, Finished, False)

## Безпосередньо прийом файлів

In [11]:
import json
import os
result_data = {

    "status": "Success",
    "count": 0,
    "error": None    
}
#print("file_raw=" + file_raw)
counter=0

all_files=[]

try:



    print(f"Отримано файлів для обробки: {file_cnt}")
    df_queue = spark.sql("SELECT subject, id FROM ev2_file_grn WHERE processing_status = 'NEW' ORDER BY processed_at ASC")
    file_list = df_queue.collect()
    if not file_list:
        print("Черга порожня. Обробка не потрібна.")
        result_data["status"] ="Success"

        mssparkutils.notebook.exit(json.dumps(result_data))

    for item in file_list:
        file_path = item["subject"]
        context_id = item["id"]        
        
        print(f"Обробка файлу: {file_path}")
        # Тут ваша логіка копіювання та UPDATE
        raw_path = file_path.strip()
        # Видаляємо початковий слеш, якщо він заважає (як ви помітили з "Files")
        if raw_path.startswith("/"):
            source_path = raw_path[1:]
        else:
            source_path = raw_path
        
        # Шлях до папки для снепшотів
        snapshots_dir = "Files/processing_snapshots"
        print("Перевіряємо, чи існує папка, і створюємо її, якщо ні")
        if not mssparkutils.fs.exists(snapshots_dir):
            print(f"Створюю папку: {snapshots_dir}")
            mssparkutils.fs.mkdirs(snapshots_dir)
        else:
            print(f"каталог {snapshots_dir}  присутній")

        print("Копіюю файл для прийому")
        if source_path:
            print( "Створюємо ім'я для тимчасової копії, щоб уникнути конфліктів")
            file_name = os.path.basename(source_path)
            snapshot_path = f"{snapshots_dir}/snap_{file_name}"
            
            print(f"Копіюю з !{source_path}! в !{snapshot_path}!")
            # Створюємо Snapshot
            mssparkutils.fs.cp( source_path, snapshot_path)
            
            # Тепер працюємо з snapshot_path — він гарантовано не зміниться!
            print(f"Починаємо обробку ізольованої копії: {snapshot_path}")
            all_files.append( {
                    "file_name": file_name,
                    "id": context_id,
                    "source_path": source_path,
                    "snapshot_path": snapshot_path
                }
            )

        counter+=1

    print("Прочитую всі файли")  
    read_files(all_files)
    print("Прочитую всі файли-ok")   

    print(f"Відмічаю, що файл оброблено") 
    set_status_processed(all_files)
    print(f"Відмічаю, що файл оброблено - ok") 

    print("Починаємо архівування файлів...")
    archive_processed_files(all_files)
    print("Архівування завершено успішно.")

    result_data["status"]="Success"     
    result_data["count"]=counter
except Exception as e:
    print(f"Помилка обробки : {e}")
    result_data["status"] = "Error"
    result_data["error"] = str(e)
  
mssparkutils.notebook.exit(json.dumps(result_data))  


StatementMeta(, 2b4638ce-0775-4bb2-86aa-df0a72e35642, 15, Finished, Available, Finished, False)

Отримано файлів для обробки: 2
Обробка файлу: /Files/terminals/19683_2026-01-19.json
Перевіряємо, чи існує папка, і створюємо її, якщо ні
каталог Files/processing_snapshots  присутній
Копіюю файл для прийому
Створюємо ім'я для тимчасової копії, щоб уникнути конфліктів
Копіюю з !Files/terminals/19683_2026-01-19.json! в !Files/processing_snapshots/snap_19683_2026-01-19.json!
Починаємо обробку ізольованої копії: Files/processing_snapshots/snap_19683_2026-01-19.json
Обробка файлу: /Files/terminals/18666_2025-05-11.json
Перевіряємо, чи існує папка, і створюємо її, якщо ні
каталог Files/processing_snapshots  присутній
Копіюю файл для прийому
Створюємо ім'я для тимчасової копії, щоб уникнути конфліктів
Копіюю з !Files/terminals/18666_2025-05-11.json! в !Files/processing_snapshots/snap_18666_2025-05-11.json!
Починаємо обробку ізольованої копії: Files/processing_snapshots/snap_18666_2025-05-11.json
Обробка файлу: /Files/terminals/14611_2025-06-23.json
Перевіряємо, чи існує папка, і створюємо її